# Detección de Amenazas Internas: Análisis de Comportamiento de Usuarios

## Fundamentos del Análisis de Comportamiento (UBA)

El **User Behavior Analytics** detecta cambios anómalos en patrones de usuarios,
identificando cuentas comprometidas y amenazas internas. Se establecen perfiles estadísticos
base y se detectan desviaciones significativas.

### Técnicas implementadas
- **Perfiles estadísticos**: media, desviación estándar, percentiles por usuario
- **Scoring de anomalía**: Z-score, distancia a la media histórica
- **Clustering**: K-Means para patrones de comportamiento
- **Detección de reglas**: horas inusuales, IPs nuevas, volumen anómalo


### Generación del Dataset de Actividad de Usuarios


In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

usuarios = ['ana.garcia', 'carlos.lopez', 'maria.torres', 'juan.perez', 'lucia.diaz']
ips_corporativas = ['10.0.1.10', '10.0.1.11', '10.0.1.12', '10.0.1.20', '10.0.2.30']
ips_externas = ['45.33.32.156', '185.220.101.5', '91.219.236.10', '104.18.5.6']

registros = []
for usuario in usuarios:
    # Cada usuario tiene un patrón horario propio
    hora_base = np.random.uniform(7.5, 9.5)   # hora típica de login
    hora_std = np.random.uniform(0.3, 1.0)
    bytes_base = np.random.uniform(5000, 50000)
    bytes_std = np.random.uniform(2000, 15000)
    ips_habituales = np.random.choice(ips_corporativas, size=2, replace=False).tolist()

    n_normal = 200
    for _ in range(n_normal):
        registros.append({
            'user':          usuario,
            'hora_login':    round(np.clip(np.random.normal(hora_base, hora_std), 0, 23.99), 2),
            'ip_origen':     np.random.choice(ips_habituales),
            'bytes_sent':    int(np.clip(np.random.normal(bytes_base, bytes_std), 100, 500000)),
            'failed_logins': np.random.choice([0, 0, 0, 0, 1], p=[0.7, 0.1, 0.1, 0.05, 0.05]),
        })

    # Inyectar eventos sospechosos (5-10 por usuario)
    n_sospechoso = np.random.randint(5, 11)
    for _ in range(n_sospechoso):
        registros.append({
            'user':          usuario,
            'hora_login':    round(np.random.uniform(0.5, 5.0), 2),         # madrugada
            'ip_origen':     np.random.choice(ips_externas),                 # IP externa
            'bytes_sent':    int(np.random.uniform(300000, 900000000)),       # volumen alto
            'failed_logins': np.random.randint(3, 15),                       # muchos fallos
        })

df_logs = pd.DataFrame(registros).sample(frac=1, random_state=42).reset_index(drop=True)
df_logs.to_csv('data/user_activity_logs.csv', index=False)

print(f"[OK] Dataset simulado guardado: data/user_activity_logs.csv ({len(df_logs)} registros)")
print(f"     Usuarios: {df_logs['user'].nunique()}")
print(f"     Registros por usuario:")
print(df_logs['user'].value_counts().to_string())
df_logs.head()

### Funciones para Perfilado y Detección de Anomalías


In [ ]:
# Listing 6.1: Modelado de comportamiento de usuario con estadísticas

import pandas as pd
import numpy as np
from scipy import stats

def construir_perfil_usuario(logs: pd.DataFrame,
                             usuario: str) -> dict:
    """
    Construye un perfil estadístico del usuario a partir
    de registros de actividad (logs).
    """
    datos_usuario = logs[logs['user'] == usuario]

    perfil = {
        'usuario':              usuario,
        'hora_media_login':     datos_usuario['hora_login'].mean(),
        'hora_std_login':       datos_usuario['hora_login'].std(),
        'ip_frecuentes':        datos_usuario['ip_origen'].value_counts()
                                .head(5).index.tolist(),
        'bytes_enviados_media': datos_usuario['bytes_sent'].mean(),
        'bytes_enviados_std':   datos_usuario['bytes_sent'].std(),
        'accesos_fallidos_p95': datos_usuario['failed_logins']
                                .quantile(0.95),
        'total_registros':      len(datos_usuario),
    }

    return perfil


def detectar_comportamiento_anomalo(evento: dict,
                                    perfil: dict,
                                    umbral_z: float = 3.0) -> list:
    """
    Compara un evento con el perfil histórico.
    Devuelve lista de alertas generadas.
    """
    alertas = []

    # Hora de inicio de sesión fuera del rango habitual
    if perfil['hora_std_login'] > 0:
        z_hora = abs(
            (evento['hora_login'] - perfil['hora_media_login'])
            / perfil['hora_std_login']
        )
        if z_hora > umbral_z:
            alertas.append(
                f"Inicio de sesión en hora inusual "
                f"(z={z_hora:.2f}): {evento['hora_login']}h"
            )

    # Acceso desde IP desconocida
    if evento['ip_origen'] not in perfil['ip_frecuentes']:
        alertas.append(
            f"Acceso desde IP no habitual: {evento['ip_origen']}"
        )

    # Volumen de datos elevado
    if (perfil['bytes_enviados_std'] > 0 and
        evento['bytes_sent'] >
        perfil['bytes_enviados_media'] + umbral_z
        * perfil['bytes_enviados_std']):
        alertas.append(
            f"Volumen de datos inusual: {evento['bytes_sent']} bytes"
        )

    return alertas


print("[OK] Funciones UBA definidas:")
print("     - construir_perfil_usuario(logs, usuario)")
print("     - detectar_comportamiento_anomalo(evento, perfil, umbral_z)")

### Construcción de Perfiles por Usuario


In [ ]:
# Cargar logs
logs = pd.read_csv('data/user_activity_logs.csv')

# Construir perfil para cada usuario
usuarios = logs['user'].unique()
perfiles = {}

print("=== Perfiles de Usuario ===")
print(f"{'Usuario':<20} {'Hora media':<12} {'Hora std':<10} {'Bytes media':<15} {'Registros':<10}")
print("-" * 67)

for usuario in usuarios:
    perfil = construir_perfil_usuario(logs, usuario)
    perfiles[usuario] = perfil
    print(
        f"{perfil['usuario']:<20} "
        f"{perfil['hora_media_login']:<12.2f} "
        f"{perfil['hora_std_login']:<10.2f} "
        f"{perfil['bytes_enviados_media']:<15.0f} "
        f"{perfil['total_registros']:<10}"
    )

print(f"\n[OK] Perfiles construidos para {len(perfiles)} usuarios.")

### Prueba: Detección de Comportamiento Anómalo


In [ ]:
# Ejemplo: evento sospechoso para ana.garcia
perfil_ana = perfiles['ana.garcia']

evento_sospechoso = {
    'hora_login':  3.5,           # 3:30 AM
    'ip_origen':   '45.33.32.156',
    'bytes_sent':  500_000_000    # 500 MB
}

alertas = detectar_comportamiento_anomalo(evento_sospechoso, perfil_ana)
if alertas:
    print(f"[!] ALERTAS para '{perfil_ana['usuario']}':")
    for alerta in alertas:
        print(f"    ⚠ {alerta}")
else:
    print("Comportamiento dentro del perfil habitual.")

print("\n--- Evento normal para comparación ---")

evento_normal = {
    'hora_login':  perfil_ana['hora_media_login'],
    'ip_origen':   perfil_ana['ip_frecuentes'][0],
    'bytes_sent':  int(perfil_ana['bytes_enviados_media'])
}

alertas_normal = detectar_comportamiento_anomalo(evento_normal, perfil_ana)
if alertas_normal:
    print(f"[!] ALERTAS para '{perfil_ana['usuario']}':")
    for alerta in alertas_normal:
        print(f"    ⚠ {alerta}")
else:
    print(f"[OK] Comportamiento de '{perfil_ana['usuario']}' dentro del perfil habitual.")

### Evaluación en Lote: Análisis del Conjunto de Logs


In [ ]:
# Evaluar cada registro del log contra el perfil del usuario correspondiente
resultados = []

for _, row in logs.iterrows():
    usuario = row['user']
    perfil = perfiles[usuario]
    evento = {
        'hora_login':  row['hora_login'],
        'ip_origen':   row['ip_origen'],
        'bytes_sent':  row['bytes_sent'],
    }
    alertas = detectar_comportamiento_anomalo(evento, perfil)
    resultados.append({
        'user':        usuario,
        'hora_login':  row['hora_login'],
        'ip_origen':   row['ip_origen'],
        'bytes_sent':  row['bytes_sent'],
        'num_alertas': len(alertas),
        'es_anomalo':  1 if len(alertas) > 0 else 0,
        'detalle':     ' | '.join(alertas) if alertas else 'Normal',
    })

df_resultados = pd.DataFrame(resultados)

print("=== Resumen de Detección UBA ===")
print(f"Total registros evaluados: {len(df_resultados)}")
print(f"Registros normales:        {(df_resultados['es_anomalo'] == 0).sum()}")
print(f"Registros anómalos:        {(df_resultados['es_anomalo'] == 1).sum()}")
print(f"\nAnómalos por usuario:")
print(df_resultados[df_resultados['es_anomalo'] == 1].groupby('user')['es_anomalo'].count())

---

## Visualización e Interpretación de Resultados


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribución de hora de login por usuario
sns.boxplot(data=logs, x='user', y='hora_login', ax=axes[0], palette='Set2')
axes[0].set_title('Distribución de hora de login por usuario')
axes[0].set_xlabel('Usuario')
axes[0].set_ylabel('Hora de login')
axes[0].tick_params(axis='x', rotation=25)

# 2. Bytes enviados: normal vs anómalo
df_plot = df_resultados.copy()
df_plot['tipo'] = df_plot['es_anomalo'].map({0: 'Normal', 1: 'Anómalo'})
sns.boxplot(data=df_plot, x='tipo', y='bytes_sent', ax=axes[1], palette=['steelblue', 'red'])
axes[1].set_title('Bytes enviados: Normal vs Anómalo')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Bytes enviados')

# 3. Alertas por usuario
alertas_por_usuario = df_resultados[df_resultados['es_anomalo'] == 1].groupby('user')['es_anomalo'].count()
alertas_por_usuario.plot(kind='bar', ax=axes[2], color='coral')
axes[2].set_title('Número de alertas por usuario')
axes[2].set_xlabel('Usuario')
axes[2].set_ylabel('Alertas')
axes[2].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('data/uba_analysis.png', dpi=150)
plt.show()

In [ ]:
# Scatter plot: hora vs bytes con anomalías resaltadas
plt.figure(figsize=(10, 6))

normales = df_resultados[df_resultados['es_anomalo'] == 0]
anomalos = df_resultados[df_resultados['es_anomalo'] == 1]

plt.scatter(normales['hora_login'], normales['bytes_sent'],
            s=8, alpha=0.3, c='steelblue', label='Normal')
plt.scatter(anomalos['hora_login'], anomalos['bytes_sent'],
            s=40, alpha=0.8, c='red', marker='x', label='Anómalo')

plt.xlabel('Hora de login')
plt.ylabel('Bytes enviados')
plt.title('Detección de comportamiento anómalo (UBA)')
plt.legend()
plt.tight_layout()
plt.savefig('data/uba_scatter.png', dpi=150)
plt.show()

---

## Persistencia de Modelos de Comportamiento


In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

# Guardar perfiles de usuario
joblib.dump(perfiles, 'models/user_profiles.pkl')
print("[OK] Perfiles de usuario guardados en models/user_profiles.pkl")

# Guardar resultados del análisis UBA
df_resultados.to_csv('data/uba_results.csv', index=False)
print("[OK] Resultados UBA guardados en data/uba_results.csv")

print("\n=== Resumen de artefactos generados ===")
print("Datos:    data/user_activity_logs.csv")
print("          data/uba_results.csv")
print("Modelo:   models/user_profiles.pkl")
print("Gráficos: data/uba_analysis.png")
print("          data/uba_scatter.png")